# Токенайзер

In [1]:
from nltk.tokenize import TweetTokenizer

In [2]:
from string import punctuation

In [3]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [4]:
from nltk.corpus import stopwords

In [5]:
from sklearn.feature_extraction.text import CountVectorizer

In [6]:
def tokenizer(t):
  t = t.lower()
  tokeniz = TweetTokenizer()
  tokeny = tokeniz.tokenize(t)
  tokeny = [tok for tok in tokeny if (tok not in stopwords.words('russian')) and (tok not in punctuation)]
  return tokeny

In [7]:
from nltk.stem.snowball import SnowballStemmer

In [8]:
st = SnowballStemmer('russian')

In [48]:
tok = tokenizer('Попытка сделать хакатон по ТП')

In [49]:
final = [st.stem(x) for x in tok]
final

['попытк', 'сдела', 'хакатон', 'тп']

# Чтение файлов из папки для выявления ключевых слов, прогнанных через токенайзер


In [14]:
def search_for_sender(file):
  for line in file:
    if 'From' in line or 'from' in line or 'Ot kogo' in line or 'От кого' in line:
      words = line.split()
      for mail in words:
        if '@' in mail:
          if '<' in mail and '>' in mail:
            return mail[1:-1]
          else:
            return mail

Извлечение из письма отправителя письма

In [13]:
def search_for_recipient(file):
  for line in file:
    if 'to' in line or 'Komu' in line or 'Кому' in line or 'To' in line:
      words = line.split()
      for mail in words:
        if '@' in mail:
          return mail
  else:
    return None

Извлечение из письма получателя письма

In [66]:
def search_subject(file):
  k = 0
  for line in file:
    if 'Subject' in line or 'subject' in line:
      words = line.split()
      k+=1
    if k == 1:
      return line

In [75]:
def search_text(file):
  text = ''
  for line in file:
    sender = search_for_sender(file)
    recipient = search_for_recipient(file)
    subject = search_subject(file)
    sender_valid = sender not in line
    recipient_valid = recipient is None or recipient not in line
    if sender_valid and recipient_valid:
      text += f'{line}'
  return text

search_text(content1)

Извлечение текста из письма, нужного для определения категории, в которую будет определено письмо


In [183]:
categories_weights = {
    'Спам': {
        'вложен': 0.15, 'выигра': 0.07,
        'приз': 0.07, 'iphone': 0.07,
        'exclusive': 0.07, 'offer': 0.07,
        'limited': 0.05, 'личност': 0.05,
        'заблокирова': 0.05, 'перейд': 0.05,
        'ссылк': 0.05, 'password-reset': 0.05,
        'secure-login': 0.05, 'time': 0.03,
        'верификац': 0.03, 'аккаунт': 0.03,
        'сброс': 0.03, 'парол': 0.03,
        'стран': 0.03, 'активн': 0.01
    },
    'Важное': {
        'urgent': 0.12, 'срочн': 0.12,
        'критическ': 0.12, 'массов': 0.09,
        'сбо': 0.12, 'работ': 0.08,
        'остановл': 0.09, 'отвеча': 0.08,
        'недоступ': 0.08, 'утечк': 0.05,
        'дан': 0.02, 'краж': 0.03
    },
    'Технические ошибки': {
        'ошибк': 0.07, 'error': 0.07,
        'err': 0.07, 'зависа': 0.06,
        'слома': 0.06, 'ddos': 0.05,
        'ремонт': 0.05, 'перебо': 0.05,
        'интернет': 0.04, 'связ': 0.04,
        'запуска': 0.04, 'обновлен': 0.04,
        'открыва': 0.04, 'компьютер': 0.03,
        'ноутбук': 0.03, 'принтер': 0.03,
        'сканер': 0.03, 'outlook': 0.03,
        'chrome': 0.03, 'excel': 0.03,
        'zoom': 0.03, 'электричеств': 0.03,
        'помех': 0.02, 'мыш': 0.02,
        'гарнитур': 0.02
    },
    'Информация': {
        'созвон': 0.15, 'дайджест': 0.11,
        'мониторинг': 0.11, 'healthcheck': 0.11,
        'дем': 0.11, 'отчет': 0.08,
        'info': 0.08, 'планов': 0.06,
        'брифинг': 0.06, 'брейншторм': 0.05,
        'обновлен': 0.04, 'встреч': 0.02,
        'работ': 0.02
    },
    'Документы': {
        'договор': 0.12, 'акт': 0.12,
        'закрыва': 0.15, 'документ': 0.15,
        'согласован': 0.15, 'contract': 0.15,
        'финальн': 0.10, 'верс': 0.03,
        'дан': 0.03
    },
    'Счета': {
        'счет': 0.25, 'оплат': 0.25,
        'invoice': 0.15, 'реквизит': 0.10,
        'задолжен': 0.03, 'акт': 0.03,
        'выписк': 0.03, 'кред': 0.03,
        'выплат': 0.03, 'бухгалтер': 0.03,
        'страховк': 0.03, 'зарплат': 0.02,
        'прем': 0.01, 'компенсац': 0.01
    },
    'Подтверждение доступа к аккаунту': {
        'vpn': 0.12, 'gitlab': 0.12,
        'confluence': 0.12, '1c': 0.12,
        'выда': 0.09, 'прав': 0.09,
        'восстанов': 0.08, 'запрос': 0.07,
        'доступ': 0.07, 'логин': 0.05,
        'аккаунт': 0.04, 'почт': 0.04,
        'парол': 0.03
    },
    'HR': {
        'отпуск': 0.10, 'больничн': 0.10,
        'резюм': 0.09, 'оформлен': 0.08,
        'должност': 0.08, 'назначен': 0.08,
        'повышен': 0.08, 'отпускн': 0.08,
        'опоздан': 0.07, 'нетрудоспособн': 0.07,
        'график': 0.05, 'сотрудник': 0.05,
        'нов': 0.04, 'работ': 0.02,
        'период': 0.01
    }
}

Разбиение слов, прогнанных через токенайзер по весу слов. Сумма слов в категории дает 1.00, что дает равные шансы для попадания письма в каждую категорию, что дает более качественный результат.


In [129]:
def classify_email(text, weights_dict):
    z = []
    if not text or len(text) == 0:
        return "Спам"

    tokens = tokenizer(text)

    if len(tokens) == 0:
        return "Спам"

    scores = {category: 0.0 for category in weights_dict.keys()}

    for token in tokens:
        stemmed_token = st.stem(token)
        z.append(stemmed_token)
        for category, words_weights in weights_dict.items():
            if stemmed_token in words_weights:
                scores[category] += words_weights[stemmed_token]

    if max(scores.values()) == 0:
        return "Не определено"

    return max(scores, key=scores.get)

In [184]:
with open('/content/HAKATON/inbox/mail_0064.txt', 'r', encoding='utf-8') as file:
    content1 = file.readlines()
    print(content1)

['От кого: no-reply <no-reply@monitoring.internal>\n', 'Кому: it-support@company.ru\n', 'Дата: 26.01.2025 15:39\n', 'Тема: Срочно подтвердите личность\n', '\n', 'Здравствуйте!\n', '\n', 'ТОЛЬКО СЕГОДНЯ! Эксклюзивная акция для сотрудников компании. Скидка 90% на все товары.\n', '\n', 'Спасибо.\n', '\n', 'Во вложении: screenshot.png']


In [185]:
classify_email(search_text(content1),categories_weights)

'Спам'